# Fake News Detection with Graph Neural Networks
Date: 2026

**Kurzbeschreibung**:
Dieses Notebook enthält die Erforschungen der Features des testGNN.ipynb Notebook, der checkpoint wurde exportiert und hier importiert mittels simple_gnn.pt Datei.

**Quellen**:
* https://distill.pub/2021/gnn-intro/

## Data Preparation

In [1]:
import torch
import os
import sys
from torch_geometric.explain import Explainer
from torch_geometric.explain.algorithm import GNNExplainer
from SimpleGNN import SimpleGNN
root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_dir not in sys.path:
    sys.path.append(root_dir)
from src.utils.data_loader import FNNDataset
import warnings
warnings.filterwarnings("ignore")

In [2]:
root = os.path.join(root_dir, "data")
device = torch.device("cpu")
dataset = FNNDataset(root=root, name="gossipcop", feature="spacy")
feature_dim = dataset.num_node_features
model = SimpleGNN(in_channels=feature_dim, hidden_channels=64, out_channels=2)
model.load_state_dict(torch.load("simple_gnn.pt"))
model = model.to(device).eval()

## Feature Explanation mittels Explainer
Mittels des Explainer von Pytorch geometric können wir die Features einiger Graphen untersuchen. Als Beispiel hier die raw values aller Features in einem sample Graphen. Die Werte beschreiben wieviel Einfluss ein feature in dem Netwerk hat, diese sind hier alle Positiv da User Features Importance normalisiert sind auf [0...1]

In [3]:
for idx in dataset.test_idx:
    data = dataset[idx]
    data.y = data.y.long()
# Build explainer
explainer = Explainer(
    model=model,  # your trained GNN
    algorithm=GNNExplainer(epochs=200),
    explanation_type='model',
    node_mask_type='attributes',  # feature importance
    edge_mask_type='object',      # edge importance
    model_config=dict(
        mode="binary_classification",    # <— this is required!
        task_level='graph',       # graph-level predictions
        return_type='raw',     # or 'log_probs' if you use log_softmax
    ),
)

# Choose one test graph
data = dataset[dataset.val_idx[7]].to(device)
if data.batch is None:
    data.batch = torch.zeros(data.x.size(0), dtype=torch.long)
# Run explainer
explanation = explainer(
    x=data.x,
    edge_index=data.edge_index.long(),
    batch=data.batch.long()
)

# Extract and print average feature importance
feat_importance = explanation.node_mask.mean(dim=0).cpu().numpy()
print("Feature importance scores:", feat_importance)


Feature importance scores: [0.24383281 0.2658138  0.2741223  0.20614578 0.30024624 0.24316368
 0.2905244  0.24032678 0.29657435 0.6208001  0.3203496  0.29007003
 0.22964537 0.19939806 0.25383908 0.27791923 0.23432067 0.5020307
 0.33138964 0.19886178 0.27230737 0.178407   0.2097011  0.26689884
 0.2882746  0.22053571 0.2974443  0.24735023 0.23200046 0.22352715
 0.2638044  0.2682787  0.21315607 0.2148898  0.22642416 0.3623861
 0.44915345 0.1839372  0.2699835  0.18929909 0.25249192 0.25124124
 0.21524604 0.23530596 0.21576181 0.37737152 0.20255028 0.23825504
 0.34969932 0.2307046  0.35463387 0.47260708 0.24021372 0.26215872
 0.30381247 0.25879052 0.23833199 0.23914874 0.27269718 0.27373073
 0.32693803 0.22040968 0.24144611 0.45462686 0.22188568 0.25464326
 0.2681794  0.20387344 0.3297667  0.27557704 0.27917856 0.211759
 0.3016424  0.22545746 0.22532076 0.26301488 0.31672314 0.19036129
 0.22823206 0.2901019  0.24229197 0.33260542 0.41602322 0.3778735
 0.2484536  0.24658091 0.42223734 0.3543

In [4]:
import numpy as np

feat_importance = explanation.node_mask.mean(dim=0).cpu().numpy()
top_idx = np.argsort(feat_importance)[-5:][::-1]
print("Top 5 important features:", top_idx)
print("Values:", feat_importance[top_idx])


Top 5 important features: [  9 140 234  17 229]
Values: [0.6208001  0.6112672  0.5166865  0.5020307  0.47695312]


## Untersuchung Nodes
Unsere Feautures sind jeweils die Node Features (genauer im UserEDA.ipynb), jeweils pro network haben diese Node feature verschieden Einfluss in das Resultat. Da wir diese Features per node haben können wir auch determinieren welche Nodes am wichtigsten sind.

In [5]:
num_node_features = data.x.size(1)
num_nodes = data.x.size(0)

# Flattened node-feature importance vector
node_mask = explanation.node_mask  # shape (num_nodes, num_features)

# If explainer flattened it, you might need to reshape
if node_mask.dim() == 1:
    node_mask = node_mask.view(num_nodes, num_node_features)
node_importance = node_mask.sum(dim=1).cpu().numpy() 
top_nodes = np.argsort(node_importance)[::-1]  # descending order
print("Node ranking by importance:", top_nodes)
print("Importance values:", node_importance[top_nodes])

Node ranking by importance: [ 0 11  8 27 26 28 25 20  4 15  6  2 18 13 17 16  1 19 14 21  9  3 22 24
 23  5 12  7 10]
Importance values: [152.31921  130.02359  120.29078  103.60675   99.46697   92.45078
  91.18093   85.37381   85.335106  84.28317   80.15185   77.66126
  76.540886  76.32893   75.869576  75.00581   70.1081    67.01083
  65.625496  65.05205   63.841713  62.896214  61.718197  61.634315
  59.74151   58.209682  55.557465  53.22095   50.321484]


Unüberraschend hier ist das Node 0 (features des Artikel) hier am wichtigsten ist

## Verständniss Feature => Node Mapping
Hier wird kurz erläutert das wir 10 features haben somit können wir FeatureID % 10 nutzen um das relevante Feature zu bestimmen und Node //10 um die Relevante NodeID zu bestimmen.

In [6]:
from pathlib import Path
from scipy.sparse import csr_matrix

data_dir = Path("../data/gossipcop")   # adjust if needed
profile_npz = data_dir / "new_profile_feature.npz" 
data_npz = np.load(profile_npz)
csr = csr_matrix((data_npz['data'], data_npz['indices'], data_npz['indptr']),
                 shape=tuple(data_npz['shape']))

# Convert to dense if needed (careful if very large!)
features_dense = csr.toarray()  # shape: (num_nodes, num_features)
print(features_dense.shape)
print(features_dense[0]) 

(314262, 10)
[0.00000000e+00 0.00000000e+00 9.21642556e-05 1.50720357e-03
 3.78178813e-02 1.14929895e-03 7.21930146e-04 6.92439414e-01
 1.10042736e-01 1.15384618e-01]


In [7]:
import pickle
import numpy as np
from scipy.sparse import csr_matrix

# Load CSR feature matrix
profile_npz = "../data/gossipcop/new_profile_feature.npz"
data_npz = np.load(profile_npz)
csr = csr_matrix((data_npz['data'], data_npz['indices'], data_npz['indptr']),
                 shape=tuple(data_npz['shape']))
features_dense = csr.toarray()  # shape: (num_nodes, num_features)

with open("../data/raw/gos_id_twitter_mapping.pkl", "rb") as f:
    node_to_twitter = pickle.load(f)

# Example: map top explainer index to node/feature
num_node_features = features_dense.shape[1]

top_idx = 86          
node_id = top_idx // num_node_features
feature_id = top_idx % num_node_features

twitter_id = node_to_twitter[node_id]
feature_value = features_dense[node_id, feature_id]

print(f"Explainer index {top_idx} → Node {node_id}, Feature {feature_id}")
print(f"Twitter ID: {twitter_id}")
print(f"Feature value: {feature_value}")


Explainer index 86 → Node 8, Feature 6
Twitter ID: 909386104561963008
Feature value: 0.0007219301457483268


In [8]:
top_idx = [105, 10, 86, 141, 157]
num_node_features = features_dense.shape[1]  # from CSR matrix

for idx in top_idx:
    node_id = idx // num_node_features
    feature_id = idx % num_node_features
    value = features_dense[node_id, feature_id]
    twitter_id = node_to_twitter[node_id]
    print(f"Explainer index {idx} → Node {node_id}, Twitter ID {twitter_id}, "
          f"Feature {feature_id}, Value {value}")


Explainer index 105 → Node 10, Twitter ID 2530389144, Feature 5, Value 7.880785801717138e-07
Explainer index 10 → Node 1, Twitter ID 199874723, Feature 0, Value 0.0
Explainer index 86 → Node 8, Twitter ID 909386104561963008, Feature 6, Value 0.0007219301457483268
Explainer index 141 → Node 14, Twitter ID 2530389144, Feature 1, Value 0.0
Explainer index 157 → Node 15, Twitter ID 2530389144, Feature 7, Value 0.6232876777648926


## Untersuchung Importance über Mehrere Netzwerke

Hier untersuchen wir falls das Netzwerk sich auf manche Features spezialisiert. Dies macht intuitiv Sinn, "Nutzer A retweeted oft Fake News, Newsposts retweeted von ihm sind Fake News". Jedoch treffen wir hier somit auf Data Leakage auf, da wir eigentlich versuchen auf Strukturelle Metriken zu fitten und nicht auf spezifische Nutzer da wir die Erscheinung dieser Nutzer nicht garantieren können.

Wir können dies Umgehen indem wir andere Datensets nutzen mit stark anderen Nutzer als test Daten nützen. Ebenfalls könnten wir auch prävalante Nutzer masken oder mittels Dropout das Modell robuster bauen. Dies würde jedoch mit einem neueren Netzwerk untersucht werden.

In [9]:
from collections import Counter

top_features_counter = Counter()
top_users_counter = Counter()

# Adjust if your central article node is not index 0
central_node_index = 0  

for i, data in enumerate(dataset):
    # Optional: limit number of graphs for testing
    if i >= 100: break

    # Prepare batch tensor for single graph
    if data.batch is None:
        data.batch = torch.zeros(data.x.size(0), dtype=torch.long)
    
    data.x = data.x.float()
    data.edge_index = data.edge_index.long()

    # Run explainer
    explanation = explainer(
        x=data.x,
        edge_index=data.edge_index,
        batch=data.batch
    )

    # Flatten node_mask if needed
    node_mask = explanation.node_mask
    if node_mask.dim() == 1:
        node_mask = node_mask.view(data.x.size(0), -1)

    # Exclude central node from user importance
    user_indices = [idx for idx in range(data.x.size(0)) if idx != central_node_index]
    user_node_importance = node_mask[user_indices].sum(dim=1)
    dominant_user_local_idx = user_node_importance.argmax().item()
    dominant_user_idx = user_indices[dominant_user_local_idx]

    # Track Twitter ID
    top_users_counter[node_to_twitter[dominant_user_idx]] += 1

    # Top feature for dominant user
    top_feature_idx = node_mask[dominant_user_idx].argmax().item()
    top_features_counter[top_feature_idx] += 1

# Inspect results
print("Most frequently dominant users (Twitter IDs):")
for user_id, count in top_users_counter.most_common(10):
    print(user_id, count)

print("\nMost frequently dominant features:")
for feat_idx, count in top_features_counter.most_common(10):
    print(feat_idx, count)

Most frequently dominant users (Twitter IDs):
2530389144 32
199874723 10
2471499625 6
785523426119213056 4
3157125579 4
4889543384 3
839519945180524544 3
20587567 3
839815575538827264 3
713385903461367809 3

Most frequently dominant features:
86 13
211 5
207 4
140 4
178 4
12 4
164 4
224 3
97 3
189 3


Wie man hier sieht, Features des Nutzer 2530389144 kommen 32/100 Netzwerken als top dominante Features vor, insbesondere 86 (listed_count (number of lists a user is part of)) 